# OpenAI Python SDK — Complete Guide

**What you'll learn:**

| # | Topic | Key Concept |
|---|-------|-------------|
| 1 | Setup & Configuration | API keys, environment variables, client init |
| 2 | Basic Completions | Responses API, `output_text` |
| 3 | Streaming | Real-time token-by-token output |
| 4 | Multi-Turn Conversations | `previous_response_id` for context chaining |
| 5 | Structured Outputs | Pydantic models + `responses.parse()` |
| 6 | Function Calling (Tools) | Giving LLMs access to external functions |
| 7 | Reasoning Mode | Chain-of-thought with `effort` control |
| 8 | Web Search | Built-in web search tool |
| 9 | Vision | Image understanding |
| 10 | RAG (File Search) | Retrieval-Augmented Generation with vector stores |
| 11 | Using Ollama (Local Models) | OpenAI-compatible API with local LLMs |

... 

**Prerequisites:** Python 3.11+, `openai` SDK, basic Python knowledge  
**API Docs:** https://platform.openai.com/docs/api-reference

In [13]:
print("hello")

hello


---
## 1. Setup & Configuration

### Install Dependencies
```bash
uv sync
```

### Environment Variables (`.env` file)

...

> **Security Best Practice:** Never hard-code API keys. Always use `.env` files and add `.env` to your `.gitignore`.

In [7]:
import os
from dotenv import load_dotenv
from pydantic import BaseModel  # handle structured data
import pandas as pd

# Load environment variables (optional)
load_dotenv()

True

### Initialize the Client

The `OpenAI()` client reads `OPENAI_API_KEY` from your environment automatically.  
No need to pass it explicitly — this is the recommended approach.

In [2]:
# client = OpenAI()
from openai import OpenAI 

client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"  # dummy key for local
)

# Equivalent explicit version (useful if you manage multiple keys):
# client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

---
## 2. Basic Completions (Responses API)

The **Responses API** (`client.responses.create`) is OpenAI's latest interface.  
It uses a REST POST request under the hood — you send input, get a response.

In [4]:
response = client.responses.create(
    model="qwen3-vl:4b",
    input="What is RAG (Retrieval-Augmented Generation)? Explain in 2-3 sentences.",
)

print(response.output_text)

RAG (Retrieval-Augmented Generation) is a technique that enhances large language models (LLMs) by first retrieving relevant information from a large external knowledge base or document corpus using a retrieval system (like vector search), then feeding that retrieved context to the LLM for generation. This process improves factual accuracy and reduces hallucinations by grounding the model's output in specific, retrieved sources. Essentially, it combines the strengths of retrieval systems (accessing up-to-date or domain-specific information) with generative AI (creating coherent responses) to produce more reliable and contextually accurate answers.


**Key parameters:**
- `model` — Which model to use (e.g., `gpt-4.1-nano`, `gpt-4.1-mini`, `gpt-4.1`)
- `input` — Your prompt (string or message list)
- `instructions` — System-level instructions (like a system prompt)

**Accessing the result:**
- `response.output_text` — The model's text reply
- `response.id` — Unique response ID (used for multi-turn conversations)
- `response.usage` — Token usage breakdown

---
## 3. Streaming Responses

Streaming delivers tokens as they're generated — essential for real-time UIs and chatbots.  
Set `stream=True` and iterate over the event stream.

In [5]:
response = client.responses.create(
    model="qwen3-vl:4b",
    instructions="You are a concise Python tutor.",
    input="Explain list comprehensions with one example.",
    stream=True,
)

for event in response:
    if event.type == "response.output_text.delta":
        print(event.delta, end="", flush=True)

print()  # Newline after streaming completes

### List Comprehensions Explained (with Example)

**What it is:**  
A concise way to create lists in one line, combining *looping* and *filtering* operations.

**Syntax:**  
`[expression for item in iterable if condition]`  

**Example:**  
Create a list of **squares** of numbers from 1 to 5:  
```python
squares = [x * x for x in range(1, 6)]
```

**Output:**  
`[1, 4, 9, 16, 25]`  

### Breakdown:
1. `x * x`: **Expression** (calculate square)  
2. `for x in range(1, 6)`: **Iterable** (loop through numbers 1-5)  
3. (Optional) `if condition`: Filter items (e.g., `if x % 2 == 0` for even numbers).  

### Why it's useful:  
- **Shorter** than `for` loops (3 lines vs. 5+ lines).  
- **Reads like English** (e.g., "square each number from 1 to 5").  
- **Efficient** (avoids manual `.append()` calls).  

> 💡 **Tip**: Use `list()` if you need a list (e.g., `list(range(10))`), but list comprehensions *are* lists by default.


**How streaming works:**
- The response object becomes an iterator of Server-Sent Events (SSE)
- `response.output_text.delta` events carry incremental text chunks
- `flush=True` forces immediate console output (no buffering)
- Other event types include `response.created`, `response.completed`, etc.

---
## 4. Multi-Turn Conversations

Chain responses using `previous_response_id` — the API automatically carries forward
the conversation context without you having to manage message history manually.

In [6]:
# Turn 1: Ask a question
response_1 = client.responses.create(
    model="gemma3:1b",
    instructions="You are a coding assistant. Be concise.",
    input="What is a Python generator?",
)
print("Turn 1:", response_1.output_text[:200], "...\n")

# Turn 2: Follow up — the model remembers Turn 1
response_2 = client.responses.create(
    model="gemma3:1b",
    previous_response_id=response_1.id,  # Links to the prior turn
    input="Show me a practical example of one.",
)
print("Turn 2:", response_2.output_text)

Turn 1: A Python generator is a special type of function that *doesn't* store its results in memory. Instead, it *yields* one value at a time as needed.  It's memory-efficient, especially for large sequences. ...

Turn 2: Okay, let's break down a practical example of a **"Personalized Learning Path Generator"** – a system that crafts customized learning experiences for individuals based on their needs, goals, and current knowledge.  This is a complex topic, but we'll focus on a simplified version for illustrative purposes.

**1. The Problem:**

* **One-Size-Fits-All Education is Ineffective:** Traditional education often assumes everyone learns at the same pace and in the same way.
* **Lack of Individualized Support:** Many learners struggle with areas where they need more support, while others may be ready to move ahead quickly.
* **Time Constraints:** Individuals have different time commitments and learning styles.

**2. The Solution: "LearnWise" – A Simplified Prototype**

LearnWise

**Why this matters:**
- No need to manually track and resend the full message history
- The server stores context server-side, linked by response IDs
- You can branch conversations by referencing different `previous_response_id` values

---
## 5. Structured Outputs with Pydantic

Use `responses.parse()` with a Pydantic model to get **guaranteed structured JSON** output.  
This is powerful for building pipelines where downstream code needs a predictable schema.

In [6]:
class CodeReview(BaseModel):
    """Schema for structured code review output."""
    language: str
    summary: str
    issues: list[str]
    severity: str  # e.g., "low", "medium", "high"

In [7]:
response = client.responses.parse(
    model="gpt-5.4-nano",
    instructions="Review the given code. Respond in the structured format provided.",
    input="Review this code: def add(a, b): return a + b",
    text_format=CodeReview,
)

review = response.output_parsed  # This is a CodeReview instance
print(f"Language : {review.language}")
print(f"Summary  : {review.summary}")
print(f"Issues   : {review.issues}")
print(f"Severity : {review.severity}")

Language : Python
Summary  : The function correctly adds two values, but it lacks basic readability/formatting improvements and input validation (depending on requirements).
Issues   : ['Code is written in a single line; formatting into multiple lines improves readability and maintainability.', 'No type hints or input validation; if non-numeric inputs are passed, behavior may be unexpected (e.g., string concatenation).', 'No tests or docstring; adding a brief docstring and/or tests would clarify intent and expected behavior.']
Severity : Low


### Convert to DataFrame for further analysis

In [8]:
df = pd.DataFrame([review.model_dump()])
df

,language,summary,issues,severity
0,Python,"The function correctly adds two values, but it...",[Code is written in a single line; formatting ...,Low


In [9]:
# Export to CSV for record-keeping
df.to_csv("code_review_results.csv", index=False)
print("Saved to code_review_results.csv")

Saved to code_review_results.csv


---
## 6. Function Calling (Tools)

Function calling lets the model decide **when** to invoke your functions and **what arguments** to pass.  
The model doesn't execute the function — it returns a structured tool call that your code handles.

### Flow:
1. Define tools (function schemas) and send them with the request
2. The model decides whether to call a tool based on the user input
3. If it does, you execute the function locally and return the result
4. The model generates a final response using the function output

In [10]:
# import json

# Step 1: Define the tool schema
tools = [
    {
        "type": "function",
        "name": "get_current_weather",
        "description": "Get the current weather for a given location.",
        "parameters": {
            "type": "object",
            "properties": {
                "location": {
                    "type": "string",
                    "description": "City and state, e.g. 'San Francisco, CA'",
                },
                "unit": {
                    "type": "string",
                    "enum": ["celsius", "fahrenheit"],
                },
            },
            "required": ["location", "unit"],
        },
    }
]

In [11]:
# Step 2: Send a message that SHOULD trigger the tool
response = client.responses.create(
    model="gpt-5.4-nano",
    tools=tools,
    input="What's the weather like in Lahore?",
    tool_choice="auto",  # "auto" = model decides | "required" = must call a tool
)

# Step 3: Check if the model made a tool call
for item in response.output:
    if item.type == "function_call":
        print(f"Function : {item.name}")
        print(f"Arguments: {item.arguments}")
    elif item.type == "message":
        print(f"Message  : {item.content[0].text}")

Function : get_current_weather
Arguments: {"location":"Lahore, PK","unit":"celsius"}


In [12]:
# When the input doesn't need a tool, the model responds normally
response = client.responses.create(
    model="gpt-5.4-nano",
    tools=tools,
    input="Hi, how are you?",
    tool_choice="auto",
)

print(response.output_text)  # Regular text response — no tool called

Hi! I’m doing well—thanks for asking. 😊 How are you doing today?


---
## 7. Reasoning Mode

Enable chain-of-thought reasoning for complex problems.  
The `effort` parameter controls how much "thinking" the model does before answering.

In [13]:
response = client.responses.create(
    model="gpt-5.4-nano",
    input="A farmer has 17 sheep. All but 9 run away. How many sheep does he have left?",
    reasoning={
        "effort": "high"  # Options: "low", "medium", "high"
    },
)

print(response.output_text)

“All but 9” means all **except** 9 run away, so the farmer has **9 sheep** left.


**When to use reasoning:**
- Math and logic problems
- Multi-step planning tasks
- Complex code generation
- Tasks where accuracy matters more than speed

> **Cost note:** Higher reasoning effort uses more tokens. Use `"low"` for simple tasks.

---
## 8. Web Search

Give the model access to live web data using the built-in `web_search_preview` tool.  
Useful for questions about recent events, current data, or anything beyond the training cutoff.

In [14]:
response = client.responses.create(
    model="gpt-5.4-nano",
    tools=[{"type": "web_search_preview"}],
    input="What are the top AI research papers this week?",
)

print(response.output_text)

“Top AI research papers this week” depends a lot on **what signal** you mean by “top” (most-discussed on social, most-viewed on arXiv, highest-citation/impact, best curated by a newsletter, etc.)—and different trackers publish different lists.

From what I can reliably verify right now, I can point you to *live weekly/top-paper aggregators* (which update daily/weekly). For example:

- **AIPapers.ai** (has “Weekly AI/ML Research Highlights”) ([aipapers.ai](https://aipapers.ai/?utm_source=openai))  
- **AI Daily PhD** (“Trending Papers This Week”) ([aidaily.phd](https://aidaily.phd/?utm_source=openai))  
- **DAIR.AI / ML-Papers-of-the-Week** (weekly top ML papers repo, curated) ([gitextract.com](https://gitextract.com/dair-ai/ML-Papers-of-the-Week?utm_source=openai))  
- **AgentScout “ArXiv cs.AI Weekly Papers Tracker”** (weekly arXiv cs.AI tracking for a specific sub-area) ([agentscout.live](https://agentscout.live/tech/ai-agents/data/arxiv-cs-ai-weekly/?utm_source=openai))  

If you te

---
## 9. Vision (Image Understanding)

Send images alongside text for multimodal understanding.  
Images can be URLs or base64-encoded data.

In [15]:
response = client.responses.create(
    model="gpt-5.4-mini",
    input=[
        {
            "role": "user",
            "content": [
                {
                    "type": "input_text",
                    "text": "Describe what you see in this image.",
                },
                {
                    "type": "input_image",
                    "image_url": "https://upload.wikimedia.org/wikipedia/commons/thumb/3/3a/Cat03.jpg/1200px-Cat03.jpg",
                },
            ],
        }
    ],
)

print(response.output_text)

A close-up of an orange tabby cat looking directly at the camera. The cat has golden eyes, a pink nose, and prominent white whiskers. The background is softly blurred, with a light-colored surface and a couple of red lines visible.


### Using local images (base64)
```python
import base64

with open("photo.jpg", "rb") as f:
    image_data = base64.standard_b64encode(f.read()).decode("utf-8")

response = client.responses.create(
    model="gpt-4.1-mini",
    input=[{
        "role": "user",
        "content": [
            {"type": "input_text", "text": "What is in this image?"},
            {
                "type": "input_image",
                "image_url": f"data:image/jpeg;base64,{image_data}",
            },
        ],
    }],
)
```

---
## 10. RAG — File Search (Retrieval-Augmented Generation)

RAG lets the model search through your uploaded documents before answering.  
OpenAI handles the vector store, chunking, and retrieval — you just point to a store ID.

### Setup Steps (done once via the API or dashboard):
1. Create a vector store
2. Upload files to the store
3. Reference the store ID in your requests

> **Note:** Replace the `vector_store_ids` and `api_key` below with your own.

In [16]:
# Use a separate client if your RAG setup uses a different key/project
rag_client = OpenAI(
    api_key=os.environ.get("OPENAI_API_KEY"),  # Use your key from .env
)

response = rag_client.responses.create(
    model="gpt-5.4-mini",
    tools=[
        {
            "type": "file_search",
            "vector_store_ids": ["vs_69d52f2677b48191a2a67f26"],
            "max_num_results": 20,
        }
    ],
    input="What social media links does the SAI company have?",
)

print(response.output_text)

The SAI company lists these social media links in its profile: GitHub, Instagram, LinkedIn, X (Twitter), Facebook, and YouTube 

Links:
- GitHub: https://github.com/the-schoolofai/
- Instagram: https://www.instagram.com/the_schoolofai/
- LinkedIn: https://www.linkedin.com/company/the-schoolof-artificial-intelligence/
- X (Twitter): https://x.com/the_schoolofai
- Facebook: https://www.facebook.com/the.school.of.artificial.intelligence
- YouTube: https://www.youtube.com/@theSchool-of-AI 


**How File Search works under the hood:**
1. Your query is embedded into a vector
2. The vector store is searched for semantically similar chunks
3. The top `max_num_results` chunks are injected into the model's context
4. The model answers using both the retrieved context and its general knowledge

---
## 11. Using Ollama (Local Models)

Ollama exposes an **OpenAI-compatible API**, so you can use the same `openai` SDK  
to talk to local models. Just change the `base_url` and `api_key`.

### Setup
```bash
# Install Ollama: https://ollama.com
ollama pull llama3.2       # Download a model
ollama serve               # Start the server (default: port 11434)
```

In [17]:
ollama_client = OpenAI(
    api_key=os.environ.get("OLLAMA_API_KEY", "ollama"),  # Ollama ignores the key
    base_url=os.environ.get("BASE_URL", "http://localhost:11434/v1"),
)

model_name = os.environ.get("LLM_MODEL", "llama3.2")

In [18]:
# Structured output with a local model via Ollama
response = ollama_client.responses.parse(
    model=model_name,
    instructions="Review the given code. Respond in the structured format provided.",
    input="Review this code: def add(a, b): return a + b",
    text_format=CodeReview,
)

review = response.output_parsed
print(f"Language : {review.language}")
print(f"Severity : {review.severity}")
print(f"Summary  : {review.summary}")

Language : python
Severity : N/A
Summary  : This is a simple function in Python that takes two parameters, `a` and `b`, and returns their sum.


---
## Quick Reference

| Feature | Method | Key Parameter |
|---------|--------|---------------|
| Basic completion | `responses.create()` | `model`, `input` |
| System prompt | `responses.create()` | `instructions` |
| Streaming | `responses.create()` | `stream=True` |
| Multi-turn | `responses.create()` | `previous_response_id` |
| Structured output | `responses.parse()` | `text_format=PydanticModel` |
| Function calling | `responses.create()` | `tools=[...]`, `tool_choice` |
| Reasoning | `responses.create()` | `reasoning={"effort": "high"}` |
| Web search | `responses.create()` | `tools=[{"type": "web_search_preview"}]` |
| Vision | `responses.create()` | `input_image` in content list |
| File search (RAG) | `responses.create()` | `tools=[{"type": "file_search", ...}]` |
| Local models | `OpenAI(base_url=...)` | Point to Ollama/vLLM/etc. |

---
*SAI - Happy building!*